# 🎬 AI 紀錄片導演工作台｜Colab G4 96GB 一鍵啟動 ComfyUI（MiniMax H3 + RealESRGAN）

> **零商業影片 API 費・100% 開源組件・全自動首尾幀接力生片**
>
> 使用前請確認上方選單 **`Runtime` ➔ `Change runtime type`** 已選擇 **`G4 GPU`**（96GB VRAM，強烈推薦）或 **`A100 GPU`**（80GB）。
>
> **使用方式只要 2 步：**
> 1. 點選上方選單 **`Runtime` ➔ `Run all`**（首次下載模型約 3～5 分鐘）。
> 2. 捲到最下方 Cell 4，複製印出的 **`https://xxxx.trycloudflare.com`** 連線網址，貼進你 Mac 本機的「AI 紀錄片導演工作台」即可 100% 全自動一句話出片！
>
> 💡 **省算力小提醒**：全部拍完後，記得點右上角 `▾` ➔ **`Disconnect and delete runtime`** 停止扣計算單元。

In [ ]:
# @title ① 檢查 Colab GPU（確認 G4 96GB / A100）與磁碟空間
import shutil, subprocess

print("=== GPU 資訊 ===")
res = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError("❌ 尚未掛載 GPU！請點上方選單 Runtime -> Change runtime type -> 選擇 G4 GPU 或 A100 GPU 後重新執行。")
print(res.stdout.strip())

total, used, free = shutil.disk_usage("/content")
print(f"\n=== 磁碟空間 ===\n可用空間：{free / (1024**3):.1f} GB / 總空間：{total / (1024**3):.1f} GB")
if free / (1024**3) < 60:
    print("⚠️ 提醒：可用磁碟空間低於 60GB，建議確認沒有殘留大型檔案。")
else:
    print("✅ 硬體環境檢查通過，準備部署 ComfyUI 與 MiniMax H3！")

In [ ]:
# @title ② 安裝最新官方 ComfyUI ＋ VideoHelperSuite ＋ RealESRGAN 超分模型
import os, subprocess
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")

if not COMFY_DIR.exists():
    print("📦 正在下載最新官方 ComfyUI...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY_DIR)], check=True)
else:
    print("🔄 ComfyUI 目錄已存在，更新至最新版本...")
    subprocess.run(["git", "-C", str(COMFY_DIR), "pull"], check=False)

print("📦 正在安裝 ComfyUI、comfy-kitchen（INT8/NVFP4 加速核心）與高速下載依賴套件...")
subprocess.run(["pip", "install", "-q", "-r", str(COMFY_DIR / "requirements.txt")], check=True)
subprocess.run(["pip", "install", "-q", "--upgrade", "comfy-kitchen", "huggingface_hub[hf_transfer]", "imageio-ffmpeg", "opencv-python-headless"], check=True)

# 安裝 ComfyUI-VideoHelperSuite（支援高效影片讀寫與 RealESRGAN 逐幀超分）
vhs_dir = COMFY_DIR / "custom_nodes" / "ComfyUI-VideoHelperSuite"
if not vhs_dir.exists():
    print("📦 安裝 ComfyUI-VideoHelperSuite...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git", str(vhs_dir)], check=True)
    subprocess.run(["pip", "install", "-q", "-r", str(vhs_dir / "requirements.txt")], check=False)

# 下載 RealESRGAN_x2plus.pth（將 1280x704 超解析度放大為清晰 1920x1080）
upscale_dir = COMFY_DIR / "models" / "upscale_models"
upscale_dir.mkdir(parents=True, exist_ok=True)
esrgan_path = upscale_dir / "RealESRGAN_x2plus.pth"
if not esrgan_path.exists():
    print("⬇️ 正在下載 RealESRGAN_x2plus.pth（約 64 MB）...")
    subprocess.run([
        "curl", "-L", "-o", str(esrgan_path),
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"
    ], check=True)

print("✅ ComfyUI 與 RealESRGAN 超分組件安裝完成！")

In [ ]:
# @title ③ 下載官方開源 MiniMax H3 權重（自動適配 G4 96GB / A100 80GB / L4 24GB）
import os, subprocess
from pathlib import Path
from huggingface_hub import hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# @markdown 是否同時下載 8-Step Turbo 加速 LoRA（L4 24GB 強烈建議勾選，大幅加速生片）
DOWNLOAD_TURBO_LORA = True  # @param {type:"boolean"}

gpu_out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
is_blackwell = any(k in gpu_out.upper() for k in ("PRO 6000", "5090", "B200", "GB", "BLACKWELL"))
text_encoder_file = (
    "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
    if is_blackwell
    else "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors"
)
print(f"🎯 偵測到 GPU：{gpu_out} ➔ 自動選用編碼器：{Path(text_encoder_file).name}")

COMFY_MODELS = Path("/content/ComfyUI/models")
REPO_ID = "Comfy-Org/MiniMax-H3"

FILES_TO_FETCH = [
    ("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", COMFY_MODELS / "diffusion_models"),
    (text_encoder_file, COMFY_MODELS / "text_encoders"),
    ("vae/minimax_h3_video_vae_fp16.safetensors", COMFY_MODELS / "vae"),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", COMFY_MODELS / "vae"),
]

if DOWNLOAD_TURBO_LORA:
    FILES_TO_FETCH.append(
        ("loras/minimax_h3_fl2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors", COMFY_MODELS / "loras")
    )

for hf_path, target_dir in FILES_TO_FETCH:
    target_dir.mkdir(parents=True, exist_ok=True)
    filename = Path(hf_path).name
    final_file = target_dir / filename
    if final_file.exists() and final_file.stat().st_size > 10_000_000:
        print(f"✅ 已存在，跳過下載：{filename}")
        continue
    print(f"⬇️ 正在從 Hugging Face 下載：{hf_path} ...")
    try:
        downloaded = hf_hub_download(
            repo_id=REPO_ID,
            filename=hf_path,
            local_dir=str(COMFY_MODELS),
        )
        print(f"   完成 -> {downloaded}")
    except Exception as exc:
        if hf_path.startswith("loras/"):
            print(f"⚠️ 選配加速 LoRA 下載略過（不影響主模型正常生片）：{exc}")
        else:
            raise

print("\n🎉 MiniMax H3 官方開源權重（主模型 + Qwen3-VL 編碼器 + 視訊/立體聲 VAE）全部就緒！")

In [ ]:
# @title ④ 啟動 ComfyUI 伺服器並取得免註冊 HTTPS 連線網址（貼到你 Mac 本機導演工作台！）
import os, re, subprocess, time, urllib.request
from pathlib import Path

# 安裝 cloudflared（完全免費、免帳號、免 Token 的官方加密通道）
cf_bin = Path("/usr/local/bin/cloudflared")
if not cf_bin.exists():
    print("🌐 下載 Cloudflare Tunnel 工具...")
    subprocess.run([
        "curl", "-L", "-o", str(cf_bin),
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ], check=True)
    cf_bin.chmod(0o755)

# 先終止舊的 ComfyUI 或 cloudflared（若重複執行本格）
subprocess.run(["pkill", "-f", "ComfyUI/main.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(1)

mem_mb = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip().splitlines()[0])
cmd = ["python", "/content/ComfyUI/main.py", "--listen", "0.0.0.0", "--port", "8188", "--preview-method", "none"]
if mem_mb >= 70000:
    cmd.append("--highvram")
print(f"🚀 正在背景啟動 ComfyUI（顯存 {mem_mb/1024:.0f} GB）...")
comfy_log = open("/content/comfyui.log", "w")
comfy_proc = subprocess.Popen(
    cmd,
    cwd="/content/ComfyUI",
    stdout=comfy_log,
    stderr=subprocess.STDOUT,
)

# 等待 ComfyUI 就緒
ready = False
for i in range(90):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    print("❌ ComfyUI 啟動逾時，最後 40 行日誌如下：")
    print(Path("/content/comfyui.log").read_text()[-2500:])
    raise RuntimeError("ComfyUI 啟動失敗")

print("✅ ComfyUI 已在本機 8188 埠啟動完成！正在建立外部安全連線網址...")
cf_proc = subprocess.Popen(
    [str(cf_bin), "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
start_t = time.time()
while time.time() - start_t < 30:
    line = cf_proc.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError("無法取得 Cloudflare Tunnel 網址，請重新執行此格。")

print("\n" + "=" * 72)
print("🎉 Colab G4 雲端生片引擎已就緒！（MiniMax H3 + RealESRGAN 1080p）")
print("👉 請複製下面這串網址，貼到你 Mac 本機『AI 導演工作台』的連線欄位：")
print(f"\n   {public_url}\n")
print("=" * 72)
print("💡 保持此儲存格執行中即可（生片期間請勿關閉此分頁）；全部拍完後記得點右上角中斷連線省算力！")

# 維持前台執行以保持 Tunnel 與 ComfyUI 存活，同時定期輸出心跳
try:
    while comfy_proc.poll() is None:
        time.sleep(30)
except KeyboardInterrupt:
    print("🛑 已手動停止 ComfyUI 服務。")
finally:
    cf_proc.terminate()
    comfy_proc.terminate()
